# Tolling agreement on Castejon I (CCGT) - Pieza 4

Valuation of a tolling agreement over Iberdrola's **Castejon I** combined-cycle
gas turbine (Navarra) by **dynamic-programming dispatch optimisation** over the
spark-spread trajectory, averaged across Monte Carlo power paths composed from
Pieza 1 (spot model) and Pieza 2 (Schwartz-Smith forward).

Pricer: `src/mibel_derivatives/products/tolling.py`. Tests:
`pytest tests/products/test_tolling.py`.

**Spark spread** (per MWh, at power P):

$$\mathrm{spark} = \text{power} - \frac{HR(P)}{3.6}\,\text{gas}_{\text{PVB}}
- \frac{HR(P)}{3.6}\,\epsilon_{CO_2}\,\text{EUA}$$

with the GJ/MWh heat rate converted to MWh_th/MWh_e by dividing by 3.6 so it
multiplies the EUR/MWh gas (MIBGAS PVB) and EUR/t carbon (EUA primary auction)
prices; $\epsilon_{CO_2}$ is the natural-gas combustion factor (tCO2/MWh_th).

The dispatch DP honours the technical band (Pmin/Pmax), minimum up/down time
(TMO/TMA), the ramp rate, and state-dependent (hot/warm/cold) start-up costs,
with heat-rate degradation at part load. It is solved with **perfect foresight
per path** - the standard deterministic-equivalent value and an upper bound on
the non-anticipative (Longstaff-Schwartz) recourse value.

In [ ]:
from __future__ import annotations

import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from mibel_derivatives.models import spot
from mibel_derivatives.products import tolling as T

pd.set_option('display.float_format', '{:.3f}'.format)
plt.rcParams['figure.dpi'] = 110

## 1. Asset specification (Castejon I)

Public reference parameters (Iberdrola Environmental Declaration 2024;
Aurecon/AEMO and NREL technical benchmarks). None is contractual.

In [ ]:
asset = T.AssetParameters()
spec = pd.DataFrame(
    [
        ('Gross design power Pmax', asset.pmax_mw, 'MW'),
        ('Technical minimum Pmin', asset.pmin_mw, 'MW'),
        ('Heat rate full load', asset.heat_rate_full_gj_per_mwh, 'GJ/MWh'),
        ('Heat rate at Pmin', asset.heat_rate_min_gj_per_mwh, 'GJ/MWh'),
        ('Start-up hot', asset.startup_cost_hot_eur_per_mw, 'EUR/MW'),
        ('Start-up warm', asset.startup_cost_warm_eur_per_mw, 'EUR/MW'),
        ('Start-up cold', asset.startup_cost_cold_eur_per_mw, 'EUR/MW'),
        ('Minimum up-time (TMO)', asset.min_uptime_h, 'h'),
        ('Minimum down-time (TMA)', asset.min_downtime_h, 'h'),
        ('Ramp rate', asset.ramp_mw_per_min, 'MW/min'),
        ('CO2 intensity', asset.co2_intensity_t_per_mwh_th, 'tCO2/MWh_th'),
    ],
    columns=['Parameter', 'Value', 'Unit'],
)
spec

Note the **ramp** of 8 MW/min = 480 MW/h, larger than the full operating band
(Pmax - Pmin = 266 MW), so the ramp constraint is *slack at hourly resolution*;
it binds only at finer resolution or for a slower unit (exercised in the tests
with a reduced ramp). The heat-rate curve is linear in power and degraded at
part load:

In [ ]:
p_grid = np.linspace(asset.pmin_mw, asset.pmax_mw, 50)
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].plot(p_grid, T.heat_rate_gj_per_mwh(asset, p_grid))
ax[0].set(xlabel='Power (MW)', ylabel='Heat rate (GJ/MWh)', title='Heat-rate curve')
ax[0].grid(alpha=0.3)
# per-MWh spark at Pmin vs Pmax for a sample price set
sp_pmin = T.spark_spread_per_mwh(asset, 90, 30, 70, power_mw=asset.pmin_mw)
sp_pmax = T.spark_spread_per_mwh(asset, 90, 30, 70, power_mw=asset.pmax_mw)
ax[1].bar(['Pmin', 'Pmax'], [float(sp_pmin), float(sp_pmax)], color=['#c62828', '#2e7d32'])
ax[1].set(ylabel='EUR/MWh', title='Clean spark @ (P=90, G=30, EUA=70)')
ax[1].grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 2. Load real market data (power / gas / carbon)

- **Power**: OMIE Spanish day-ahead spot (hourly), aggregated to daily baseload.
- **Gas**: MIBGAS PVB day-ahead daily product `GDAES_D+1` (Castejon pays PVB,
  not TTF).
- **Carbon**: EUA primary-auction clearing price (successful auctions),
  forward-filled to daily.

In [ ]:
omie = pd.read_parquet('data/curated/omie_spot_es_2019_2024.parquet')
omie_h = omie.set_index('datetime_utc')['price_eur_mwh'].sort_index()

pvb = pd.read_parquet('data/curated/mibgas_pvb.parquet')
gas = pvb[pvb['product_code'] == 'GDAES_D+1'].copy()
gas['d'] = pd.to_datetime(gas['first_day_delivery'])
gas_daily = gas.set_index('d')['daily_reference_price_eur_mwh'].sort_index()

eua = pd.read_parquet('data/curated/eua_primary_auction.parquet')
eua = eua[eua['status'] == 'successful'].copy()
eua['d'] = pd.to_datetime(eua['auction_date'])
eua_daily = eua.set_index('d')['clearing_price_eur_t'].sort_index()

power_daily = omie_h.resample('D').mean()
power_daily.index = power_daily.index.tz_localize(None)
idx = pd.date_range('2022-01-01', '2024-12-31', freq='D')
P = power_daily.reindex(idx).ffill()
G = gas_daily.reindex(idx).ffill()
E = eua_daily.reindex(idx).ffill()
pd.DataFrame({'power': P, 'gas_pvb': G, 'eua': E}).describe()

## 3. Historical spark spread 2022-2024

Clean spark spread at full-load heat rate. The 2022 gas crisis pushed the spread
deeply negative; it recovers through 2023-2024. The fraction of profitable days
foreshadows the dispatch capacity factor.

In [ ]:
spark = T.spark_spread_per_mwh(asset, P.values, G.values, E.values)
print(f'mean   : {np.nanmean(spark):8.2f} EUR/MWh')
print(f'min/max: {np.nanmin(spark):8.2f} / {np.nanmax(spark):8.2f} EUR/MWh')
print(f'positive days: {np.nanmean(spark > 0):.1%}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(idx, spark, lw=0.7, color='#1f4e79')
ax.axhline(0, color='k', lw=0.6, ls='--')
ax.fill_between(idx, spark, 0, where=spark > 0, color='#2e7d32', alpha=0.4)
ax.fill_between(idx, spark, 0, where=spark <= 0, color='#c62828', alpha=0.3)
ax.set(ylabel='EUR/MWh', title='Castejon I clean spark spread (full-load HR), daily 2022-2024')
fig.tight_layout()
plt.show()

## 4. Compose Pieza 1 + Pieza 2: simulate Q1 2025 power paths

The power paths are simulated from the Pieza 1 slow-fast mean-reverting
jump-diffusion spot model, calibrated on 2023-2024 OMIE and started from the
last fitted slow-factor level. In a production run the level would be anchored to
the Pieza 2 Schwartz-Smith forward curve (`models.spot.fit_with_forward_anchor`);
here the recent calibration window sets the level, so the **absolute** value
reflects 2023-2024 prices rather than a market forward for Q1 2025.

Gas and carbon enter as **deterministic forward curves** (flat at the last
30-day mean), per the spec.

In [ ]:
N_PATHS = 1000          # 1000 for this notebook; 50000 for a reported run (pod)
n_hours = 24 * 90       # Q1 2025

hist = omie_h['2023-01-01':'2024-12-31']
fit = spot.fit(hist)
theta0 = float(fit.theta_series.iloc[-1])
print(f'spot fit: {fit.n_jumps} jumps detected')

power_paths = spot.simulate(
    fit.params,
    start=pd.Timestamp('2025-01-01', tz='UTC'),
    n_hours=n_hours,
    n_paths=N_PATHS,
    initial_theta=theta0,
    seed=2025,
)
gas_f = float(G.iloc[-30:].mean())
eua_f = float(E.iloc[-30:].mean())
gas_curve = np.full(n_hours, gas_f)
eua_curve = np.full(n_hours, eua_f)
print(f'sim mean power {power_paths.mean():.1f} EUR/MWh | gas {gas_f:.1f} | eua {eua_f:.1f}')

## 5. Tolling valuation for Q1 2025

`price_tolling` runs the dispatch DP over every power path and averages the
discounted optimal gross margin, then nets the fixed capacity fee.

In [ ]:
agreement = T.TollingAgreement(asset=asset, fixed_fee_eur_per_mw_year=30_000.0)
t0 = time.time()
res = T.price_tolling(agreement, power_paths, gas_curve, eua_curve, chunk_size=250)
print(f'priced {N_PATHS} paths in {time.time() - t0:.1f}s\n')

print(f'Option value (gross dispatch margin) : {res.option_value/1e6:8.2f} EUR million')
print(f'  Monte Carlo std error              : {res.std_error/1e6:8.2f} EUR million')
print(f'Fixed fee PV                          : {res.fixed_fee_pv/1e6:8.2f} EUR million')
print(f'Net value to offtaker                 : {res.net_value/1e6:8.2f} EUR million')
print(f'Implied EUR/MW/year                   : {res.option_value/asset.pmax_mw/(n_hours/8760):,.0f}')
print()
print(f'Mean capacity factor                  : {res.mean_capacity_factor:.1%}')
print(f'Mean running hours / {n_hours}            : {res.mean_running_hours:.0f}')
print(f'Mean number of starts                 : {res.mean_n_starts:.1f}')
print(f'Mean start-up cost                    : {res.mean_startup_cost/1e6:.2f} EUR million')

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
ax[0].hist(res.per_path_value / 1e6, bins=40, color='#1f4e79', alpha=0.8)
ax[0].axvline(res.option_value / 1e6, color='r', lw=1.2, label='mean')
ax[0].set(xlabel='Per-path PV margin (EUR million)', ylabel='paths', title='Valuation distribution')
ax[0].legend()
# one sample dispatch week
sample = T.optimise_dispatch(power_paths[:1], gas_curve, eua_curve, asset)
hrs = slice(0, 24 * 7)
ax[1].plot(power_paths[0, hrs], lw=0.8, label='power price', color='#888')
axb = ax[1].twinx()
axb.fill_between(range(24 * 7), sample.power_schedule[0, hrs], step='mid', alpha=0.4,
                 color='#2e7d32', label='dispatch MW')
ax[1].set(title='Sample week: price vs dispatch', xlabel='hour')
ax[1].set_ylabel('EUR/MWh'); axb.set_ylabel('MW')
fig.tight_layout()
plt.show()

## 6. Sensitivity analysis (gas, carbon, heat rate)

Re-price under +/- 30% shocks to each driver, reusing the same power paths
(400 of them for speed). Value falls with gas and carbon (higher fuel/carbon
cost shrinks the spread) and with heat rate (worse efficiency); the heat rate is
the dominant lever.

In [ ]:
NP = 400
pp = power_paths[:NP]
shocks = np.array([-0.30, -0.15, 0.0, 0.15, 0.30])


def value_for(gas_mult=1.0, eua_mult=1.0, hr_mult=1.0):
    a = T.AssetParameters(
        heat_rate_full_gj_per_mwh=asset.heat_rate_full_gj_per_mwh * hr_mult,
        heat_rate_min_gj_per_mwh=asset.heat_rate_min_gj_per_mwh * hr_mult,
    )
    ag = T.TollingAgreement(asset=a, fixed_fee_eur_per_mw_year=30_000.0)
    r = T.price_tolling(ag, pp, np.full(n_hours, gas_f * gas_mult),
                        np.full(n_hours, eua_f * eua_mult), chunk_size=200)
    return r.option_value / 1e6


gas_sens = [value_for(gas_mult=1 + s) for s in shocks]
eua_sens = [value_for(eua_mult=1 + s) for s in shocks]
hr_sens = [value_for(hr_mult=1 + s) for s in shocks]
sens = pd.DataFrame(
    {'gas': gas_sens, 'carbon': eua_sens, 'heat_rate': hr_sens},
    index=[f'{int(s*100):+d}%' for s in shocks],
)
sens

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
x = shocks * 100
ax.plot(x, gas_sens, '-o', label='Gas (MIBGAS PVB)')
ax.plot(x, eua_sens, '-s', label='Carbon (EUA)')
ax.plot(x, hr_sens, '-^', label='Heat rate')
ax.axvline(0, color='k', lw=0.5, ls='--')
ax.set(xlabel='Shock to input (%)', ylabel='Option value (EUR million)',
       title='Tolling value sensitivity (400 paths)')
ax.legend(); ax.grid(alpha=0.3)
fig.tight_layout()
plt.show()

## 7. Summary

- The DP dispatch optimiser honours Pmin/Pmax, TMO/TMA, ramp and hot/warm/cold
  start-up costs, with degraded part-load heat rate; all constraints are
  asserted in `tests/products/test_tolling.py`.
- For Q1 2025 the unit runs roughly a third of the hours, consistent with the
  ~40% profitable-day share in the 2022-2024 history.
- The valuation is a **perfect-foresight upper bound**; the non-anticipative
  Longstaff-Schwartz recourse value (the value of foresight gap) is the
  documented extension.
- The value is most sensitive to **heat rate**, then gas, then carbon - the
  efficiency assumption dominates the toll economics.

See `reports/diagnostics/tolling.md` for the full methodology write-up.